# Notebook with code snippets for (simple) running HBV-SASK model

## This notebook contains code snippets for simply running the model and examing/plotting the output

### Originally, data comes from this source: 
* Gupta, Hoshin V. and Razavi, Saman: "Revisiting the Basis of Sensitivity Analysis for Dynamical Earth System Models", Water Resources Research, 2018
* VARS-Tool https://github.com/vars-tool/vars-tool/tree/master/src/varstool/example_models

In [ ]:
import numpy as np
import pathlib
import pandas as pd
import sys
import time

In [ ]:
from uqef_dynamic.utils import utility
from uqef_dynamic.models.hbv_sask import hbvsask_utility as hbv
from uqef_dynamic.models.hbv_sask import HBVSASKModel as hbvmodel

In [ ]:
# importing modules/libs for plotting
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

import matplotlib.pyplot as mp

from plotly.offline import plot

pd.options.plotting.backend = "plotly"

### Defining paths

In [ ]:
# TODO - change these paths accordingly
hbv_model_data_path = pathlib.Path("/work/ga45met/Hydro_Models/HBV-SASK-data")
configurationObject = pathlib.Path('/work/ga45met/mnt/linux_cluster_2/UQEF-Dynamic/data/configurations/configuration_hbv_10D_full.json')
basin = "Oldman_Basin"  # 'Banff_Basin' | 'Oldman_Basin'

inputModelDir = hbv_model_data_path

# TODO - change this path accordingly
workingDir = hbv_model_data_path / basin / "model_runs" / 'ensamble_run_full' #"whole_time_generating_state_df"


# Creating Model Object

Creating a model object

In [ ]:
writing_results_to_a_file = True
plotting = True
createNewFolder = True # create a separate folder to save results for each model run

hbvsaskModelObject = hbvmodel.HBVSASKModel(
    configurationObject=configurationObject,
    inputModelDir=inputModelDir,
    workingDir=workingDir,
    basin=basin,
    writing_results_to_a_file=writing_results_to_a_file,
    plotting=plotting
)

In [ ]:
# get to know some of the relevant time settings, read from a json configuration file
print(f"start_date: {hbvsaskModelObject.start_date}")
print(f"start_date_predictions: {hbvsaskModelObject.start_date_predictions}")
print(f"end_date: {hbvsaskModelObject.end_date}")
print(f"full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")
print(f"simulation_range is of length {len(hbvsaskModelObject.simulation_range)} days")

In [ ]:
hbvsaskModelObject.start_date

Examing the input/forcing data and ground-truth/measured data if on disposal...

In [ ]:
hbvsaskModelObject.time_series_measured_data_df

In [ ]:
hbvsaskModelObject.time_series_measured_data_df['streamflow'].values #y_hat

In [ ]:
hbvsaskModelObject.plot_input_data(read_measured_streamflow=True)

In [ ]:
hbvsaskModelObject.initial_condition_df

## Analysing initial condition file

In [ ]:
hbvsaskModelObject.initial_condition_file

In [ ]:
temp = hbv.read_initial_conditions(initial_condition_file=hbvsaskModelObject.initial_condition_file)
temp

In [ ]:
temp = hbv.read_initial_conditions(
    hbvsaskModelObject.initial_condition_file, 
    timestamp=hbvsaskModelObject.start_date,
    time_column_name=hbvsaskModelObject.time_column_name
)
temp

In [ ]:
# basin = "Banff_Basin" #"Oldman_Basin"
temp_initial_condition_file = hbv_model_data_path / basin / "state_df.pkl"
temp_initial_condition_file_const = hbv_model_data_path / basin / "state_const_df.pkl"

temp  = pd.read_pickle(temp_initial_condition_file, compression="gzip")
temp

# Running a single model run without changing the parameter values

In [ ]:
start = time.time()
results_array = hbvsaskModelObject.run(createNewFolder=createNewFolder)
end = time.time()
runtime = end - start
print(f"single execution of the model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")

## Examing the model output

Model run returns an array. \
Each element of the array is a tuple \
The first element of each tuple is a dictionary storing different info about model run \
The second element of each tuple is a runtime (which is as well stored in the above-mentioned dictionary)

In [ ]:
print(type(results_array))
print(f"len of the resulted array is equal to the total number of model runs (one set of parameters one run) - {len(results_array)}")
print(type(results_array[0]))
print(f"The first element of each tuple is a {type(results_array[0][0])}")
print(f"The second element of each tuple is a {type(results_array[0][1])}")
print(f"runtime : {results_array[0][1]}")

In [ ]:
print(f"start_date - {hbvsaskModelObject.start_date}")
print(f"end_date - {hbvsaskModelObject.end_date}")
print(f"start_date_predictions - {hbvsaskModelObject.start_date_predictions}")

In [ ]:
results_array[0]

more about the result dictionary... it stores different dataframes

In [ ]:
results_array[0][0].keys()

In [ ]:
results_array[0][0]['run_time']

In [ ]:
results_array[0][0]['parameters_dict']

pd.DataFrame storing index_run, paramter values and values for different likelihood functions/goodness-of-fit (GoF) functions

In [ ]:
results_array[0][0]['gof_df']

the output of the model is in the form of a time-series stored in a pd.DataFrame...

In [ ]:
results_array[0][0]['result_time_series']

In [ ]:
results_array[0][0]['result_time_series'].columns

plotting input, predicted/simulated and measured time-series

In [ ]:
fig = hbv.plot_streamflow_and_precipitation(
    input_data_df=hbvsaskModelObject.time_series_measured_data_df, 
    simulated_data_df=results_array[0][0]['result_time_series'], 
    input_data_time_column=hbvsaskModelObject.time_column_name,
    simulated_time_column=hbvsaskModelObject.time_column_name, 
    observed_streamflow_column=hbvsaskModelObject.streamflow_column_name,
    simulated_streamflow_column="Q_cms", 
    precipitation_columns=hbvsaskModelObject.precipitation_column_name)
fig.show()

column 'stramflow' contains the measured data, column "Q_cms" contains predicted data (i.e., streamflow expressed in cubic meters per second) by the model defined with the current values for the uncertain parameters

as a QoI, one can take Q_cms time-series (extract it from above dataframe), AET (Actual EvapoTranspiration), or some likelihood (i.e., goodness-of-fit (GoF)) function value

In [ ]:
qoi = results_array[0][0]['result_time_series']["Q_cms"].values
qoi

In [ ]:
qoi_2 = results_array[0][0]['gof_df']["RMSE"].values
qoi_2

In [ ]:
state_df = results_array[0][0]['state_df']
state_df.columns

In [ ]:
# plotting state
state_df = results_array[0][0]['state_df']
fig = go.Figure()
fig.add_trace(go.Scatter(x=state_df.index,y=state_df["SMS"],name="SMS - Soil Storage",))
fig.add_trace(go.Scatter(x=state_df.index,y=state_df["S1"], name="S1 - Fast Reservoir",))
fig.add_trace(go.Scatter(x=state_df.index,y=state_df["S2"], name="S2 - Slow Reservoir",))
fig.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=state_df.index,y=state_df["SWE"],name="SWE - Snow water equivalen",))
fig.show()

In [ ]:
fig = hbv.plot_input_output_state(
    modelObject = hbvsaskModelObject,
    result_df = results_array[0][0]['result_time_series'],
    state_df = results_array[0][0]['state_df']
)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=20,  # Left margin
        r=20   # Right margin
    )
)
fig.show()

# Running a single model run with propagating the values for uncertain parameters specified in json configuration file

One can specify the direct values of the uncertain parameters in the form of a dictionary. The order and naming of the parameters has to follow the order from the configuration_json_file_dict["parameters"] 

In [ ]:
parameter_value_dict = {'TT': -4.0, 'C0': 0.0, 'ETF': 0.0, 'FC': 50, 'FRAC': 0.1, 'K2': 0.025} # old default values
parameter_value_dict_1 = {'TT': 0.0, 'C0': 0.5, 'ETF': 0.2, 'FC': 250, 'FRAC': 0.1, 'K2': 0.025}
parameter_value_dict_2 = {'TT': 0.0, 'C0': 0.5, 'ETF': 0.2, 'FC': 250, 'FRAC': 0.1, 'K2': 0.1}
parameter_value_dict_3 = {'TT': -4.0, 'C0': 5.0, 'ETF': 0.5, 'FC': 50, 'FRAC': 0.1, 'K2': 0.025}
parameter_value_dict = {'TT': 0.0, 'C0': 0.5, 'ETF': 0.2, 'FC': 250, 'FRAC': 0.3, 'K2': 0.1, 'K1': 0.5, 'alpha':2.0}



In [ ]:
unique_run_index = 1 
start = time.time()
results_array_changed_param = hbvsaskModelObject.run(
#     i_s = [1,2,3,4],
#     parameters = [parameter_value_dict, parameter_value_dict_1, parameter_value_dict_2, parameter_value_dict_3],
    i_s = [unique_run_index,],
    parameters = [parameter_value_dict,],
    createNewFolder=createNewFolder,
    take_direct_value=True
)
end = time.time()
runtime = end - start
print(f"single execution of the model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")

In [ ]:
results_array_changed_param[0][0]['parameters_dict']

In [ ]:
results_array_changed_param[0][0]['result_time_series']

In [ ]:
results_array_changed_param[0][0]['result_time_series']['Q_cms'].values #y

In [ ]:
results_array_changed_param[0][0]['state_df']

## Examing the model output

In [ ]:
results_array_changed_param[0][0]['parameters_dict']

In [ ]:
results_array_changed_param[0][0]['result_time_series']

In [ ]:
# dataframe containing predicted state data
state_df = results_array_changed_param[0][0]['state_df']
state_df

Compare the 'newly' computed state DataFrame with the one saved for the whole time span for this basin...

In [ ]:
temp_initial_condition_file = hbv_model_data_path / basin / "state_df.pkl"
temp  = pd.read_pickle(temp_initial_condition_file, compression="gzip")
temp

In [ ]:
fig = px.line(state_df, x=state_df.index, y=['SWE',], title="SWE")
fig.show()

In [ ]:
# fig = state_df.plot(x=state_df.index, y=["SMS", "S1", "S2"], kind="line", )

fig = go.Figure()
fig.add_trace(go.Scatter(x=state_df.index,y=state_df["SMS"],name="Soil Storage",))
fig.add_trace(go.Scatter(x=state_df.index,y=state_df["S1"], name="Fast Reservoir",))
fig.add_trace(go.Scatter(x=state_df.index,y=state_df["S2"], name="Slow Reservoir",))
fig.show()

In [ ]:
error_time_series = np.array(results_array_changed_param[0][0]['result_time_series']['streamflow'].values) - np.array(results_array_changed_param[0][0]['result_time_series']['Q_cms'].values)
error_time_series



# Plotting the output and input data

In [ ]:
fig = hbv.plot_streamflow_and_precipitation(
    input_data_df=hbvsaskModelObject.time_series_measured_data_df, 
    simulated_data_df=results_array_changed_param[0][0]['result_time_series'], 
    input_data_time_column=hbvsaskModelObject.time_column_name,
    simulated_time_column=hbvsaskModelObject.time_column_name, 
    observed_streamflow_column=hbvsaskModelObject.streamflow_column_name,
    simulated_streamflow_column="Q_cms", 
    precipitation_columns=hbvsaskModelObject.precipitation_column_name)
fig.show()

In [ ]:
hbvsaskModelObject.time_series_measured_data_df.columns

In [ ]:
results_array_changed_param[0][0]['result_time_series'].columns

In [ ]:
results_array_changed_param[0][0]['state_df'].columns

In [ ]:
fig = hbv.plot_input_output_state(
    modelObject = hbvsaskModelObject,
    result_df = results_array_changed_param[0][0]['result_time_series'],
    state_df = results_array_changed_param[0][0]['state_df']
)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=20,  # Left margin
        r=20   # Right margin
    )
)
fig.show()

# Reading the saved output of the model

In [ ]:
# paths to saved output from model run
path_to_input = hbv_model_data_path / basin
i = 1 # index of model run of interest
if createNewFolder:
    flux_output_file = workingDir / f"run_{i}" / f"flux_df_{i}.pkl"
    state_output_file = workingDir / f"run_{i}" / f"state_df_{i}.pkl"
else:
    flux_output_file = workingDir / f"flux_df_{i}.pkl"
    state_output_file = workingDir / f"state_df_{i}.pkl"

In [ ]:
flux_df = pd.read_pickle(flux_output_file, compression="gzip")
flux_df

In [ ]:
state_df = pd.read_pickle(state_output_file, compression="gzip")
state_df

In [ ]:
# re-computation of some qoodnes-of-fit/likelihood functions
# flux_df = results_array_changed_param[0][0]['result_time_series']
gof_list = ["MAE", "MSE", "RMSE", "NRMSE", "NSE", "LogNSE", "KGE"]
gof_dict = utility.calculateGoodnessofFit_simple(
    measuredDF = flux_df,
    simulatedDF = flux_df,
    gof_list = gof_list,
    measuredDF_time_column_name=hbvsaskModelObject.time_column_name,
    measuredDF_column_name=hbvsaskModelObject.streamflow_column_name,
    simulatedDF_time_column_name=hbvsaskModelObject.time_column_name,
    simulatedDF_column_name='Q_cms',
    return_dict=True,
)
gof_dict

# Banff Basin - Running the model for some time period and for changed model parameters

In [ ]:
basin = "Banff_Basin"
inputModelDir = hbv_model_data_path
inputModelDir_basin = inputModelDir / basin
workingDir = inputModelDir_basin / "model_runs" / "temp_run_oct_2025"

writing_results_to_a_file = True
plotting = True
createNewFolder = True # create a separate folder to save results for each model run

# Zooming in on the period of interest
configurationObject_time = {
    "time_settings":
    {
      "start_day": 2,
      "start_month": 10,
      "start_year": 2002,
      "end_day": 1,
      "end_month": 10,
      "end_year": 2007,
      "run_full_timespan":"False",
      "spin_up_length":1096,
      "simulation_length": 700,
    },
    "simulation_settings":{
        "qoi":["Q_cms", "AET"],
        "qoi_column":["Q_cms", "AET"],
        "read_measured_data": ["True", "False"],
        "qoi_column_measured":["streamflow", "None"]
    }
}

hbvsaskModelObject_Banff_sampled = hbvmodel.HBVSASKModel(
    configurationObject=configurationObject_time,
    inputModelDir=inputModelDir,
    workingDir=workingDir,
    basin=basin,
    writing_results_to_a_file=writing_results_to_a_file,
    run_full_timespan=False, 
    plotting=plotting
)

print(f"start_date: {hbvsaskModelObject_Banff_sampled.start_date}")
print(f"start_date_predictions: {hbvsaskModelObject_Banff_sampled.start_date_predictions}")
print(f"end_date: {hbvsaskModelObject_Banff_sampled.end_date}")
print(f"full_data_range is {len(hbvsaskModelObject_Banff_sampled.full_data_range)} days including spin_up_length of {hbvsaskModelObject_Banff_sampled.spin_up_length} days")
print(f"simulation_range is of length {len(hbvsaskModelObject_Banff_sampled.simulation_range)} days")

fig = hbvsaskModelObject_Banff_sampled.plot_input_data(
    plot_whole_simulation_period=False, 
    fileName=f"forcing_data_{hbvsaskModelObject_Banff_sampled.start_date_predictions}_{hbvsaskModelObject_Banff_sampled.end_date}.html",
    title=f"{basin} {hbvsaskModelObject_Banff_sampled.start_date_predictions}-{hbvsaskModelObject_Banff_sampled.end_date}")

fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=10,  # Left margin
        r=10   # Right margin
    )
)
# plot_filename = workingDir / f"forcing_data_{hbvsaskModelObject_Banff_sampled.start_date_predictions}_{hbvsaskModelObject_Banff_sampled.end_date}.pdf"
# fig.write_image(str(plot_filename), format="pdf")
display(fig)

In [ ]:
parameter_value_dict_1 = {'TT': 0.0, 'C0': 0.5, 'ETF': 0.2, 'FC': 250, 'FRAC': 0.1, 'K2': 0.025, 'K1': 0.5, 'alpha':2.0}
parameter_value_dict_2 = {'TT': 0.0, 'C0': 0.5, 'ETF': 0.2, 'FC': 250, 'FRAC': 0.1, 'K2': 0.1, 'K1': 0.5, 'alpha':2.0}
parameter_value_dict_3 = {'TT': -4.0, 'C0': 5.0, 'ETF': 0.5, 'FC': 50, 'FRAC': 0.1, 'K2': 0.025, 'K1': 0.5, 'alpha':2.0}
parameter_value_dict = {'TT': 0.0, 'C0': 0.5, 'ETF': 0.2, 'FC': 250, 'FRAC': 0.3, 'K2': 0.1, 'K1': 0.5, 'alpha':2.0}

unique_run_index = [1,] 
unique_run_index = [1, 2, 3, 4]
start = time.time()
results_array_changed_param = hbvsaskModelObject_Banff_sampled.run(
#     i_s = [1,2,3,4],
    parameters = [parameter_value_dict, parameter_value_dict_1, parameter_value_dict_2, parameter_value_dict_3],
    i_s = unique_run_index,
    # parameters = [parameter_value_dict,],
    createNewFolder=createNewFolder,
    take_direct_value=True
)
end = time.time()
runtime = end - start
print(f"execution of the {len(unique_run_index)} model's run function take {runtime}; full_data_range is {len(hbvsaskModelObject.full_data_range)} days including spin_up_length of {hbvsaskModelObject.spin_up_length} days")

In [ ]:
len(results_array_changed_param)

In [ ]:
fig = hbv.plot_streamflow_and_precipitation(
    input_data_df=hbvsaskModelObject_Banff_sampled.time_series_measured_data_df, 
    simulated_data_df=results_array_changed_param[2][0]['result_time_series'], 
    input_data_time_column=hbvsaskModelObject_Banff_sampled.time_column_name,
    simulated_time_column=hbvsaskModelObject_Banff_sampled.time_column_name, 
    observed_streamflow_column=hbvsaskModelObject_Banff_sampled.streamflow_column_name,
    simulated_streamflow_column="Q_cms", 
    precipitation_columns=hbvsaskModelObject_Banff_sampled.precipitation_column_name)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=10,  # Left margin
        r=10   # Right margin
    )
)
plot_filename = workingDir / f"forcing_data_measured_streamflow_{hbvsaskModelObject_Banff_sampled.start_date_predictions}_{hbvsaskModelObject_Banff_sampled.end_date}.pdf"
fig.write_image(str(plot_filename), format="pdf")
plot_filename = workingDir / f"forcing_data_measured_streamflow_{hbvsaskModelObject_Banff_sampled.start_date_predictions}_{hbvsaskModelObject_Banff_sampled.end_date}.html"
plot(fig, filename=str(plot_filename), auto_open=False)
display(fig)

In [ ]:
# let's plot also state data
fig = hbv.plot_input_output_state(
    modelObject = hbvsaskModelObject_Banff_sampled,
    result_df = results_array_changed_param[2][0]['result_time_series'],
    state_df = results_array_changed_param[2][0]['state_df']
)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=20,  # Left margin
        r=20   # Right margin
    )
)
plot_filename = workingDir / f"forcing_measured_state_streamflow_{hbvsaskModelObject_Banff_sampled.start_date_predictions}_{hbvsaskModelObject_Banff_sampled.end_date}.pdf"
fig.write_image(str(plot_filename), format="pdf", height=1000, width=1100,)
plot_filename = workingDir / f"forcing_measured_state_streamflow_{hbvsaskModelObject_Banff_sampled.start_date_predictions}_{hbvsaskModelObject_Banff_sampled.end_date}.html"
plot(fig, filename=str(plot_filename), auto_open=False)
fig.show()